# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [5]:
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !mkdir -p _tools
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_models.py -o _tools/colab_models.py
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_neural_trainer.py -o _tools/colab_neural_trainer.py
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

Sat Oct 24 06:23:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             28W /   70W |    4219MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
FILE_ID = "1XnalYuDXB8ywwPw9siVNWlFKH1bbjT7O"              # Свежий датасет от 23.10 (4 окна H=25)
HORIZON = 25                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "0"                                              # Список окон: "all" или "0,1"
#ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"
ARCHS = "attention,mlp_bottleneck"                         # Архитектуры через запятую
RUNS_PER_ARCH = 15                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 5e-3                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    file_id=FILE_ID,
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

☁️ Прямое скачивание prl_neural_colab_data.npz с Google Drive (ID=1XnalYuDXB8ywwPw9siVNWlFKH1bbjT7O)...
⚡ Подготовка memory-mapped кэша на диске (экономит ~4 ГБ RAM)...

🚀 ОБУЧЕНИЕ ОКНА WIN_00 | H=25с (K=3.0, L=75с) | Train=375,000 (2026-10-17 15:34..2026-10-22 12:37)
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.002500
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.001250
   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок (6/10 эп.).
  [ATTENTION | win_00 | r01/15] 🏆 РЕКОРД (135.4с) -> Loss: 1.8136 | IC: +0.1260 | AUC: 0.6875 | Top-2%: +0.05
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.002500
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.001250
   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок (6/10 эп.).
  [ATTENTION | win_00 | r02/15] 🏆 РЕКОРД (128.8с) -> Loss: 1.5638 | IC: +0.1335 | AUC: 0.6993 | Top-2%: +0.13
   📉 [Backtrack] Откат к лучшим весам + сброс инерц

In [8]:
# 3. Выгрузка готового архива моделей (авто-упаковка при прерванной 2-й ячейке)
import os, shutil
HORIZON = globals().get("HORIZON", 25)
!pkill -9 -f "colab_neural_trainer" 2>/dev/null || true
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
ZIP_NAME = f"{PACK_DIR}.zip"
if os.path.exists(PACK_DIR):
    shutil.make_archive(PACK_DIR, "zip", root_dir=".", base_dir=PACK_DIR)
if os.path.exists(ZIP_NAME):
    print(f"📦 Архив найден: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB)")
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://x0.at/ | head -c 200 && echo " (x0.at)"
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://store1.gofile.io/uploadFile | head -c 400 && echo " (gofile)"
    !curl -s --max-time 60 -H "Accept: application/json" -T "{ZIP_NAME}" https://filebin.net/prl_h{HORIZON}s_pack/{ZIP_NAME} | head -c 300 && echo " (filebin)"

^C
📦 Архив найден: prl_ensemble_pack_h25s.zip (4.12 MB)
https://x0.at/8mRD.zip
 (x0.at)
{"data":{"code":"qvnevKLa","createTime":1792827375,"downloadPage":"https://gofile.io/d/ap9BHslj","guestToken":"UGAqWj5apMxgbQ2eCPXOTcxgEZoTx2cd","id":"d596747f-0ce5-4834-8b0f-236ad84763c5","md5":"7ef1b9491a547c7a4347e4862a883812","mimetype":"application/zip","modTime":1792827375,"name":"prl_ensemble_pack_h25s.zip","parentFolder":"c3c7d3bd-574e-491d-aa22-1dddc597bd7d","parentFolderCode":"ap9BHslj", (gofile)
{
    "bin": {
        "id": "prl_h25s_pack",
        "readonly": false,
        "bytes": 4335814,
        "bytes_readable": "4.3 MB",
        "files": 1,
        "updated_at": "2026-10-24T07:36:27.356974Z",
        "updated_at_relative": "now",
        "created_at": "2026-10-21T12:17:32.137515Z",
  (filebin)
